# Apply meteorological clustering

This notebook runs the meteorological regime classification (Cold Katabatic /
Transitional Synoptic / Warm Synoptic) by calling the `meteo_cluster` function
defined in [`meteo_cluster.py`](meteo_cluster.py), and then makes the main
plots.

All the configuration lives in **one cell** below: change the period, the
resolution, the climatology years, the sensor heights, or whether to include
wind direction, and re-run.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# make sure this folder is importable, then load our function
sys.path.insert(0, str(Path.cwd()))
from meteo_cluster import meteo_cluster

%load_ext autoreload
%autoreload 2
print('Imports OK')

## 1. Configuration

Everything you might want to change is here.

In [ ]:
# ---- analysis settings -------------------------------------------------
# Each year is clustered SEPARATELY (per-year), as in Gorodetskaya (2013),
# but the climatology (mean cycle removed) is built from ALL clean years.
START   = '2025-01-01'        # first day to classify
END     = '2025-12-31'        # last day to classify
FREQ    = '1h'                # '1h' (hourly, blowing-snow analysis) or '1d' (daily, Gorodetskaya comparison)
CLIM_YEARS = [2023, 2024, 2025]   # clean years used to build the mean cycle
INCLUDE_WD = False            # 5 Gorodetskaya variables; wind direction added a posteriori

# ---- load vs recompute -------------------------------------------------
# LOAD_SAVED = True  -> read the official per-year CSV in OFFICIAL_DIR and make
#                       the figures WITHOUT rerunning the clustering (fast, no
#                       pickle loading). The standardised anomalies needed for
#                       the heatmap/pairplot are rebuilt from the saved
#                       climatology CSVs; the dendrogram is skipped (the Ward
#                       linkage matrix is not saved).
# LOAD_SAVED = False -> run the full pipeline via meteo_cluster() (recompute).
LOAD_SAVED   = False
OFFICIAL_DIR = Path(r'D:\output_24_06')

# ---- sensor heights [m] ------------------------------------------------
# Leave as None -> per-year defaults in meteo_cluster.py (SENSOR_HEIGHTS):
# 2023: 1.30/1.63 ; 2024: 1.55/1.60 ; 2025: 1.374/1.374.
# The seasonal change of the snow level is handled automatically via HS_Cor.
H_T = None
H_S = None

# ---- paths -------------------------------------------------------------
# data_template defaults to D:\slowdata_24_06 inside meteo_cluster.py
DATA_TEMPLATE = r'D:\slowdata_24_06\slowdata_cleaned_{year}.pkl'
SRC_PATH      = r'C:\Users\lory2\Desktop\tesi\DataProcessingScripts\src'

# ad-hoc runs go to a subfolder so they DON'T overwrite the official outputs
# (the canonical per-year files in D:\output_24_06 are produced by the dedicated
#  run script and already include the a-posteriori wind direction).
OUTPUT_DIR = Path(r'D:\output_24_06')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / f'regimes_{START[:4]}_{FREQ}_retrysingle2025.csv'

## 2. Get the labelled data

Two modes (set `LOAD_SAVED` above):

- **`LOAD_SAVED = True`** — read the official per-year CSV from `OFFICIAL_DIR`
  (no clustering, no pickle loading). The standardised anomalies `X_std`
  (needed for the heatmap and pairplot) are rebuilt from the saved climatology
  CSVs; the dendrogram is skipped because the Ward linkage matrix is not saved.
- **`LOAD_SAVED = False`** — run the full pipeline with `meteo_cluster()`.

Either way the labelled table ends up in `df` and a `res`-like dict in `res`.

In [ ]:
import meteo_cluster as mc
from meteo_cluster import (_year_variables, _detrend,
                           REGIME_ORDER, REGIME_COLORS, REGIME_MARKERS)

if LOAD_SAVED:
    # ---- load the official saved CSV (no clustering) --------------------
    csv_path = OFFICIAL_DIR / f'regimes_{START[:4]}_{FREQ}.csv'
    df = pd.read_csv(csv_path, index_col='datetime', parse_dates=True)
    df = df.loc[(df.index >= pd.Timestamp(START)) & (df.index <= pd.Timestamp(END))]

    variables = [v for v in ['q', 'P', 'WS', 'LW_in', 'T_inversion', 'WD']
                 if v in df.columns and (v != 'WD' or INCLUDE_WD)]
    order   = REGIME_ORDER
    COLORS  = REGIME_COLORS
    MARKERS = REGIME_MARKERS

    # rebuild the standardised anomalies from the SAVED climatology CSVs, so the
    # heatmap / pairplot work without rerunning the clustering or loading pickles.
    from sklearn.preprocessing import StandardScaler
    seas = (pd.read_csv(OFFICIAL_DIR / 'climatology_seasonal_2023_2025.csv')
              .set_index('dayofyear'))
    diur = None
    if FREQ.endswith('h'):
        diur = (pd.read_csv(OFFICIAL_DIR / 'climatology_diurnal_2023_2025.csv')
                  .set_index(['month', 'hour']))
    anomalies = _detrend(df, (seas, diur), FREQ, variables).dropna()
    X_std = pd.DataFrame(
        StandardScaler().fit_transform(anomalies),
        index=anomalies.index, columns=variables)

    # load the saved Ward linkage matrix (.npy saved alongside the CSV)
    linkage_path = OFFICIAL_DIR / f'regimes_{START[:4]}_{FREQ}.npy'
    Z = np.load(linkage_path) if linkage_path.exists() else None
    if Z is None:
        print(f'No linkage matrix found at {linkage_path}. '
              'Run with LOAD_SAVED=False once to generate it.')

    res = {'data': df, 'variables': variables, 'regime_order': order,
           'regime_colors': COLORS, 'regime_markers': MARKERS,
           'X_std': X_std, 'linkage': Z}
    print(f'Loaded {len(df)} periods from {csv_path}')

else:
    # ---- recompute the full pipeline -----------------------------------
    res = meteo_cluster(
        start=START, end=END, freq=FREQ,
        clim_years=CLIM_YEARS,
        include_wd=INCLUDE_WD, h_T=H_T, h_S=H_S,
        data_template=DATA_TEMPLATE, src_path=SRC_PATH,
        output_csv=OUTPUT_CSV,
    )
    df        = res['data']
    variables = res['variables']
    order     = res['regime_order']
    COLORS    = res['regime_colors']
    MARKERS   = res['regime_markers']

    # wind direction a posteriori (not used in clustering when INCLUDE_WD=False),
    # joined back per year so we can describe each regime's direction.
    if 'WD' not in df.columns:
        from utils.utils import RH_to_specific_humidity as _q
        wd = pd.concat([
            _year_variables(y, FREQ, DATA_TEMPLATE, H_T, H_S, 0.99, _q, 'WS2_Avg', 'WD2')['WD']
            for y in range(pd.Timestamp(START).year, pd.Timestamp(END).year + 1)
        ])
        df['WD'] = wd[~wd.index.duplicated()].reindex(df.index)

df.head()

## 3. Dendrogram

The Ward tree; the dashed line is where it is cut to obtain the three regimes.

In [ ]:
from scipy.cluster.hierarchy import dendrogram

if res.get('linkage') is None:
    print('Linkage matrix not available. Run once with LOAD_SAVED=False to generate and save it.')
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    dendrogram(res['linkage'], no_labels=True, ax=ax)
    ax.axhline(res['linkage'][-2, 2], color='crimson', ls='--', lw=1, label='cut for k=3')
    ax.set(title=f'Ward dendrogram - {START[:4]} {FREQ}', xlabel='periods', ylabel='distance')
    ax.legend()
    plt.tight_layout(); plt.show()

## 4. Regime counts and mean meteorology

How many periods fall in each regime, and the average conditions per regime.

In [ ]:
print(df['regime'].value_counts().reindex(order), '\n')
summary = df.groupby('regime')[variables + ['TA']].mean().reindex(order)
summary.round(3)

## 5. Heatmap of standardised anomalies

Median standardised value of each variable per regime: it shows at a glance
what makes each regime different (e.g. Cold Katabatic = strong inversion,
weak wind).

In [ ]:
plot_vars = [v for v in ['LW_in', 'WS', 'P', 'q', 'T_inversion', 'WD'] if v in variables]
medians = (res['X_std'].join(df['regime'])
           .groupby('regime')[plot_vars].median().reindex(order))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(medians.T.values, cmap='jet', aspect='auto')
ax.set_xticks(range(len(order))); ax.set_xticklabels([r[:4] for r in order])
ax.set_yticks(range(len(plot_vars))); ax.set_yticklabels(plot_vars)
fig.colorbar(im, ax=ax, label='median standardised anomaly')
ax.set_title(f'Cluster medians - {START[:4]} {FREQ}')
plt.tight_layout(); plt.show()

## 6. Time series coloured by regime

Each variable over time, with points coloured by the assigned regime.

In [ ]:
ts_vars = [('WS', 'Wind speed\nm s-1'), ('TA', 'Temperature\nC'),
           ('T_inversion', 'T inversion\nK m-1'), ('LW_in', 'LW down\nW m-2')]
ts_vars = [(v, lab) for v, lab in ts_vars if v in df.columns]

fig, axes = plt.subplots(len(ts_vars), 1, figsize=(12, 2.3*len(ts_vars)), sharex=True)
for ax, (v, lab) in zip(axes, ts_vars):
    for reg in order:
        m = df['regime'] == reg
        ax.scatter(df.index[m], df.loc[m, v], c=COLORS[reg], s=5, alpha=0.7, label=reg)
    ax.set_ylabel(lab, fontsize=8)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
axes[0].legend(markerscale=3, fontsize=8, loc='upper right')
fig.suptitle(f'Time series by regime - {START[:4]} {FREQ}')
plt.tight_layout(); plt.show()

## 7. Pairplot

Pairwise relationships between the variables, coloured by regime.

In [ ]:
import seaborn as sns
g = sns.pairplot(df[plot_vars + ['regime']], hue='regime', hue_order=order,
                 vars=plot_vars, palette=COLORS, plot_kws={'s': 14, 'alpha': 0.5})
g.fig.suptitle(f'Pairplot - {START[:4]} {FREQ}', y=1.01)
plt.show()

## 8. Wind speed vs wind direction

Where each regime sits in the wind speed / direction plane.

In [ ]:
if 'WD' in df.columns:
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for reg in order:
        s = df[df['regime'] == reg]
        ax.scatter(s['WD'], s['WS'], c=COLORS[reg], marker=MARKERS[reg],
                   s=18, alpha=0.6, label=reg)
    ax.set(xlabel='Wind direction (deg)', ylabel='Wind speed (m s-1)', xlim=(0, 360))
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('Wind direction not included (INCLUDE_WD=False).')